In [1]:
import shutil
import os

# Define la ruta del archivo que deseas copiar
ruta_origen = '/kaggle/input/progress/loocv_progreso (3).json'

# Define la ruta del directorio de destino
ruta_destino = '/kaggle/working/'

# Crea el directorio de destino si no existe
os.makedirs(ruta_destino, exist_ok=True)

# Define el nuevo nombre del archivo
nuevo_nombre = 'loocv_progress.json'

# Copia el archivo al directorio de destino con el nuevo nombre
shutil.copy(ruta_origen, os.path.join(ruta_destino, nuevo_nombre))

print("Archivo copiado y renombrado con éxito.")


Archivo copiado y renombrado con éxito.


In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import LeaveOneOut
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import regularizers
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from tensorflow.keras import models
from collections import Counter
import gc
import json

# --- Configuración General ---
DATASET_PATH = '/kaggle/input/psoriasis-dataset/dataset_v6'
IMG_SIZE = 224
BATCH_SIZE = 16
EPOCHS_LOOCV = 5 # Épocas para CADA fold de LOOCV 
RANDOM_SEED = 45
LEARNING_RATE = 0.001

# --- State Files for Resuming ---
PROGRESS_FILE = '/kaggle/working/loocv_progress.json'

# --- Funciones de Preprocesamiento y Balanceo ---

def load_all_images(directory, img_size):
    classes = sorted(os.listdir(directory))
    images, labels = [], []
    class_indices = {class_name: i for i, class_name in enumerate(classes)}
    print(f"Cargando y redimensionando imágenes a {img_size}x{img_size}...")
    for class_name in classes:
        class_path = os.path.join(directory, class_name)
        if not os.path.isdir(class_path):
            continue
        for img_name in os.listdir(class_path):
            if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                img_path = os.path.join(class_path, img_name)
                try:
                    img = tf.keras.preprocessing.image.load_img(img_path, target_size=(img_size, img_size))
                    img_array = tf.keras.preprocessing.image.img_to_array(img)
                    img_array = preprocess_input(img_array) # Correcto para EfficientNet
                    images.append(img_array)
                    labels.append(class_indices[class_name])
                except Exception as e:
                    print(f"Error cargando imagen {img_path}: {e}")
    return np.array(images), np.array(labels), classes

def balance_dataset_with_augmentation(X_data, y_data, class_names_list, target_img_size):
    print(f"Balanceando {len(X_data)} muestras de entrenamiento con aumentación...")
    datagen = ImageDataGenerator(
        rotation_range=30,
        width_shift_range=0.2,
        height_shift_range=0.2,
        shear_range=0.2,
        zoom_range=0.3,
        horizontal_flip=True,
        brightness_range=[0.9, 1.1],
        channel_shift_range=30.0,
        fill_mode='nearest'
    )
    class_counts = Counter(y_data)
    if not class_counts:
        print("Advertencia: No hay datos para balancear.")
        return np.array([]), tf.keras.utils.to_categorical(np.array([]), num_classes=len(class_names_list))

    max_count = max(class_counts.values()) if class_counts else 0
    X_balanced, y_balanced = list(X_data), list(y_data)

    if not class_names_list or max_count == 0:
        print("Advertencia: No se puede realizar el balanceo.")
        num_actual_classes = len(class_names_list) if class_names_list else (np.max(y_data) + 1 if y_data.size > 0 else 1)
        return np.array(X_balanced), tf.keras.utils.to_categorical(np.array(y_balanced), num_classes=num_actual_classes)

    for class_idx in range(len(class_names_list)):
        current_indices = np.where(y_data == class_idx)[0]
        current_count = len(current_indices)
        if 0 < current_count < max_count:
            needed = max_count - current_count
            images_to_augment = X_data[current_indices]
            augmented_count = 0
            idx_img_to_augment = 0
            while augmented_count < needed:
                img_original = images_to_augment[idx_img_to_augment % len(images_to_augment)]
                img = np.expand_dims(img_original, 0)
                batch = next(datagen.flow(img, batch_size=1))
                X_balanced.append(batch[0])
                y_balanced.append(class_idx)
                augmented_count += 1
                idx_img_to_augment +=1
    num_final_classes = len(class_names_list)
    return np.array(X_balanced), tf.keras.utils.to_categorical(np.array(y_balanced), num_classes=num_final_classes)

# --- Función para Crear el Modelo (base congelada) ---
def create_efficientnet_model_frozen_base(input_shape, num_classes_model):
    base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=input_shape)
    base_model.trainable = False # Base congelada

    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(512, activation='relu', kernel_regularizer=regularizers.l2(0.001))(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)
    x = Dense(256, activation='relu', kernel_regularizer=regularizers.l2(0.001))(x)
    x = Dropout(0.2)(x)
    outputs = Dense(num_classes_model, activation='softmax')(x)

    model = Model(inputs=base_model.input, outputs=outputs)

    model.compile(optimizer=Adam(learning_rate=LEARNING_RATE),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

# --- Main Pipeline ---

print("Cargando todas las imágenes para LOOCV...")
X_all, y_all, class_names = load_all_images(DATASET_PATH, IMG_SIZE)
if X_all.size == 0:
    raise ValueError("No se cargaron imágenes.")
print(f"Total de imágenes cargadas: {len(X_all)}, Clases: {class_names}")
num_classes = len(class_names)
total_samples = len(X_all)

start_fold_idx = 0
all_true_labels_loo = []
all_pred_labels_loo = []
fold_accuracies_loo = []

if os.path.exists(PROGRESS_FILE):
    try:
        with open(PROGRESS_FILE, 'r') as f:
            progress_data = json.load(f)
        start_fold_idx = progress_data.get('next_fold_idx', 0)
        all_true_labels_loo = progress_data.get('true_labels', [])
        all_pred_labels_loo = progress_data.get('pred_labels', [])
        fold_accuracies_loo = progress_data.get('fold_accuracies', [])
        print(f"Progreso cargado. Reanudando desde el fold {start_fold_idx + 1}.")
        if start_fold_idx >= total_samples:
            print("LOOCV ya completado según el archivo de progreso.")
    except Exception as e:
        print(f"Error cargando el archivo de progreso: {e}. Empezando desde el principio.")
        start_fold_idx = 0
        all_true_labels_loo = []
        all_pred_labels_loo = []
        fold_accuracies_loo = []
else:
    print("No se encontró archivo de progreso. Empezando desde el principio.")

loo = LeaveOneOut()

if start_fold_idx < total_samples:
    print(f"\nIniciando/Reanudando Leave-One-Out Cross-Validation ({total_samples} iteraciones)...")
    print(f"Se realizarán {EPOCHS_LOOCV} épocas por fold (base del modelo congelada).")

    for current_run_idx, (train_indices, val_indices) in enumerate(loo.split(X_all)):
        fold_idx = current_run_idx
        if fold_idx < start_fold_idx:
            if fold_idx == 0: print(f"Saltando folds hasta el {start_fold_idx +1}...")
            continue

        print(f"\nProcesando Fold LOOCV {fold_idx + 1}/{total_samples}")
        X_train_loo, X_val_loo = X_all[train_indices], X_all[val_indices]
        y_train_loo, y_val_loo = y_all[train_indices], y_all[val_indices]

        X_train_loo_balanced, y_train_loo_balanced_categorical = balance_dataset_with_augmentation(
            X_train_loo, y_train_loo, class_names, IMG_SIZE
        )

        if X_train_loo_balanced.size == 0:
            print(f"Fold {fold_idx + 1} sin datos de entrenamiento tras balanceo, saltando.")
            all_true_labels_loo.append(y_val_loo[0])
            all_pred_labels_loo.append(-1) # Indicador de error
            fold_accuracies_loo.append(0)
        else:
            # Crear el modelo con la base congelada
            model_loo = create_efficientnet_model_frozen_base((IMG_SIZE, IMG_SIZE, 3), num_classes)

            model_loo.fit(
                X_train_loo_balanced, y_train_loo_balanced_categorical,
                epochs=EPOCHS_LOOCV,
                batch_size=BATCH_SIZE,
                verbose=0 # Reducir salida durante el entrenamiento de cada fold
            )

            y_pred_prob_fold = model_loo.predict(X_val_loo, verbose=0)
            y_pred_class_fold = np.argmax(y_pred_prob_fold, axis=1)[0]

            all_true_labels_loo.append(y_val_loo[0])
            all_pred_labels_loo.append(y_pred_class_fold)
            accuracy_this_fold = 1 if y_pred_class_fold == y_val_loo[0] else 0
            fold_accuracies_loo.append(accuracy_this_fold)
            print(f"  Fold {fold_idx + 1}: Verdadera={class_names[y_val_loo[0]]}, Predicha={class_names[y_pred_class_fold[0] if isinstance(y_pred_class_fold, np.ndarray) else y_pred_class_fold]}. Accuracy: {accuracy_this_fold}")


            del model_loo
            tf.keras.backend.clear_session()
            gc.collect()

        # Guardar progreso después de cada fold
        progress_data_to_save = {
            'next_fold_idx': int(fold_idx + 1), # Convertir a int nativo de Python
            'true_labels': [int(label) for label in all_true_labels_loo], # Convertir cada etiqueta
            'pred_labels': [int(label) for label in all_pred_labels_loo], # Convertir cada etiqueta
            'fold_accuracies': [float(acc) for acc in fold_accuracies_loo] # Convertir a float
        }
        try:
            with open(PROGRESS_FILE, 'w') as f:
                json.dump(progress_data_to_save, f, indent=4)
            if (fold_idx + 1) % 20 == 0: 
                 print(f"    Progreso guardado después del fold {fold_idx + 1}.")
        except Exception as e:
            print(f"    Error guardando progreso: {e}")
        
               
        # Descomenta si quieres parar en bloques de 100 para revisión manual
        # if (fold_idx + 1) % 100 == 0 and (fold_idx + 1) < total_samples :
        # print(f"Se han completado {fold_idx + 1} folds. Deteniendo para guardar y permitir la reanudación.")
        # print(f"Para continuar, vuelve a ejecutar el script.")
        # exit()


# --- Resultados Finales de LOOCV ---
print("\n--- Resultados Finales de LOOCV ---")
if not all_true_labels_loo or (start_fold_idx < total_samples and len(all_true_labels_loo) < total_samples) :
     print(f"LOOCV no completado totalmente o reanudado. Resultados basados en {len(all_true_labels_loo)}/{total_samples} folds procesados.")
elif not all_true_labels_loo :
    print("No hay resultados para analizar (all_true_labels_loo está vacía).")
    
if all_true_labels_loo: # Solo proceder si hay resultados
    valid_indices = [i for i, pred in enumerate(all_pred_labels_loo) if pred != -1] # Filtrar folds fallidos
    if not valid_indices:
        print("No se completó ninguna predicción válida en LOOCV.")
    else:
        y_true_filtered = np.array(all_true_labels_loo)[valid_indices]
        y_pred_filtered = np.array(all_pred_labels_loo)[valid_indices]
        
        # Calcular la precisión promedio solo de los folds que produjeron una predicción válida
        accuracies_filtered = np.array(fold_accuracies_loo)[valid_indices]
        mean_loocv_accuracy = np.mean(accuracies_filtered) if len(accuracies_filtered) > 0 else 0.0

        print(f"\nPrecisión Promedio de Leave-One-Out Cross-Validation: {mean_loocv_accuracy*100:.2f}%")

        # Generar matriz de confusión y reporte
        conf_matrix_loocv = confusion_matrix(y_true_filtered, y_pred_filtered, labels=list(range(num_classes)))
        plt.figure(figsize=(10, 8))
        sns.heatmap(conf_matrix_loocv, annot=True, fmt='d', cmap='Blues',
                    xticklabels=class_names, yticklabels=class_names)
        plt.title('Matriz de Confusión - LOOCV')
        plt.xlabel('Predicción')
        plt.ylabel('Real')
        plt.tight_layout()
        plt.show()

        print("\nInforme de Clasificación - LOOCV:")
        print(classification_report(y_true_filtered, y_pred_filtered, labels=list(range(num_classes)), target_names=class_names, zero_division=0))
else:
    print("No hay resultados de LOOCV para mostrar.")

print("\nScript LOOCV simplificado finalizado.")

2025-05-27 11:30:59.561606: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1748345460.037744      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1748345460.164967      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Cargando todas las imágenes para LOOCV...
Cargando y redimensionando imágenes a 224x224...
Total de imágenes cargadas: 450, Clases: ['Psoriasis Eritrodermica', 'Psoriasis Guttata', 'Psoriasis Inversa', 'Psoriasis Pustulosa', 'Psoriasis Vulgar']
Progreso cargado. Reanudando desde el fold 318.

Iniciando/Reanudando Leave-One-Out Cross-Validation (450 iteraciones)...
Se realizarán 5 épocas por fold (base del modelo congelada).
Saltando folds hasta el 318...

Procesando Fold LOOCV 318/450
Balanceando 449 muestras de entrenamiento con aumentación...


I0000 00:00:1748345489.463864      19 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13942 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1748345489.464663      19 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13942 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


I0000 00:00:1748345511.643420      63 service.cc:148] XLA service 0x7d263c0d4a50 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1748345511.645087      63 service.cc:156]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1748345511.645107      63 service.cc:156]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1748345513.732481      63 cuda_dnn.cc:529] Loaded cuDNN version 90300
I0000 00:00:1748345525.079667      63 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


  Fold 318: Verdadera=Psoriasis Vulgar, Predicha=Psoriasis Vulgar. Accuracy: 1

Procesando Fold LOOCV 319/450
Balanceando 449 muestras de entrenamiento con aumentación...
  Fold 319: Verdadera=Psoriasis Vulgar, Predicha=Psoriasis Vulgar. Accuracy: 1

Procesando Fold LOOCV 320/450
Balanceando 449 muestras de entrenamiento con aumentación...
  Fold 320: Verdadera=Psoriasis Vulgar, Predicha=Psoriasis Vulgar. Accuracy: 1
    Progreso guardado después del fold 320.

Procesando Fold LOOCV 321/450
Balanceando 449 muestras de entrenamiento con aumentación...
  Fold 321: Verdadera=Psoriasis Vulgar, Predicha=Psoriasis Vulgar. Accuracy: 1

Procesando Fold LOOCV 322/450
Balanceando 449 muestras de entrenamiento con aumentación...
  Fold 322: Verdadera=Psoriasis Vulgar, Predicha=Psoriasis Vulgar. Accuracy: 1

Procesando Fold LOOCV 323/450
Balanceando 449 muestras de entrenamiento con aumentación...
  Fold 323: Verdadera=Psoriasis Vulgar, Predicha=Psoriasis Vulgar. Accuracy: 1

Procesando Fold LOOCV